# 05 - Figure 1

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('../src'))
from src import data
from src.plotting import *   # shared figure style defaults
from src import config

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.cm as cm
import seaborn as sns

# Parameters

In [ ]:
save = True
lean = False
input_clustering = '_new' # '_new' or ''
optimal_cluster = 'st_cluster_final_gid' # 'st_cluster_final_gid' or 'st_cluster_3_5_7'

test_case = 'block'
historic_ref = '' # '_Full' '_Discount' or ''
premium_type = '' # '_Full'  '_Discount' or ''

use_reinsurance = True
base_case = ''
if use_reinsurance == False:
    base_case = '_base_case'

if lean == True:
    lean = 'Lean_'
else:
    lean = ''

In [ ]:
if test_case == 'historic':
    simulations = 1
    steps = 45
else:
    steps = 100
    simulations = 1000

# Data Load

## Simulations

In [ ]:
res = data.load_simulation_results(test_case, input_clustering, premium_type,
                                   base_case, lean=lean, historic_ref=historic_ref)
state_balance_df, final_balances_df = res['state_balance'], res['final_balances']
state_balance_df_hist = res['state_balance_hist']

# Plotting

## Timeseries of State Pools

In [ ]:
# Abbreviation mapping (if needed)
state_abbrev = config.FIPS_TO_ABBREV

# Selected state abbreviations
selected_states = ['LA', 'MS', 'ND', 'NY', 'NJ', 'TX', 'MO', 'OK', 'WV', 'AL', 'IA', 'MN', 'PA', 'FL', 'CA']

# If state_balance_df is already a DataFrame:
state_year_df_hist = state_balance_df_hist.copy()

# Normalize STATEFP to zero-padded 2-char strings
state_year_df_hist["STATEFP"] = state_year_df_hist["STATEFP"].astype(str).str.extract(r"(\d+)", expand=False).str.zfill(2)

In [ ]:
# Add abbreviation column
state_year_df_hist["STATEFP"] = state_year_df_hist["STATEFP"].astype(str).str.extract(r"(\d+)", expand=False).str.zfill(2)
state_year_df_hist["State"] = state_year_df_hist["STATEFP"].map(state_abbrev)

# Filter to selected states
filtered_df_hist = state_year_df_hist[state_year_df_hist["State"].isin(selected_states)].copy()

# Group by state and year across simulations, compute mean balance
state_balance_by_year_hist = (
    filtered_df_hist
    .groupby(["State", "year"])["contribution"]
    .sum()
    .groupby(level=0).cumsum()  # Cumulative NFIP balance by year
    .reset_index()
    .rename(columns={"contribution": "nfip_balance"})
)

In [ ]:
# If state_balance_df is already a DataFrame:
state_year_df = state_balance_df.copy()

# Normalize STATEFP to zero-padded 2-char strings
state_year_df["STATEFP"] = state_year_df["STATEFP"].astype(str).str.extract(r"(\d+)", expand=False).str.zfill(2)

# Compute stats across simulations for each state and year
state_year_df["cumulative_balance"] = (
    state_year_df
    .groupby(["simulation", "STATEFP"])["contribution"]
    .cumsum()
)

# Add abbreviation
state_year_df["State"] = state_year_df["STATEFP"].map(state_abbrev)

# Filter to selected states
filtered_df = state_year_df[state_year_df["State"].isin(selected_states)].copy()

filtered_df["cumulative_balance"] = filtered_df["cumulative_balance"]/1000000
state_balance_by_year_hist["nfip_balance"] = state_balance_by_year_hist["nfip_balance"]/1000000

# Aggregate across simulations: min, max, mean, IQR
state_balance_stats = (
    filtered_df
    .groupby(["State", "year"])["cumulative_balance"]
    .agg(
        min="min",
        max="max",
        mean="mean",
        q25=lambda x: x.quantile(0.25),
        q75=lambda x: x.quantile(0.75),
    )
    .reset_index()
)

# Subplot grid geometry for the state panels of the final figure
n_states = len(selected_states)
n_cols = 3
n_rows = -(-n_states // n_cols)

## National Pool

In [ ]:
# Summary per simulation
# Canonical list of simulations present
sim_index = pd.Index(sorted(final_balances_df['simulation'].unique()), name='simulation')

# Pooled (cumulative) balance per simulation over years
time_series_df = (
    state_balance_df
    .groupby(['simulation', 'year'], as_index=False)['nfip_balance'].sum()
    .rename(columns={'nfip_balance': 'total_balance'})
)

time_series_df_hist = (
    state_balance_df_hist
    .groupby(['simulation', 'year'], as_index=False)['nfip_balance'].sum()
    .rename(columns={'nfip_balance': 'total_balance'})
)

time_series_df['total_balance'] = time_series_df['total_balance']/1000000000
time_series_df_hist['total_balance'] = time_series_df_hist['total_balance']/1000000000

# Compute stats across simulations
stats_by_year = (
    time_series_df
    .groupby('year')['total_balance']
    .agg(
        min='min',
        max='max',
        mean='mean',
        q25=lambda x: x.quantile(0.25),
        q75=lambda x: x.quantile(0.75)
    )
    .reset_index()
    .sort_values('year')
)

In [ ]:
# Sum contributions and balances across states per year + simulation
federal_df = (
    state_balance_df
    .groupby(['simulation', 'year'], as_index=False)
    .agg({'contribution': 'sum', 'nfip_balance': 'sum'})
)

# Apply national logic
def pooled_drawdown(row):
    contrib, bal = row['contribution'], row['nfip_balance']
    if contrib >= 0:
        return 0
    elif bal >= 0:
        return abs(min(0, bal + contrib))
    else:
        return abs(contrib)

federal_df['reins_need'] = federal_df.apply(pooled_drawdown, axis=1)

# Sum across time to get total reinsurance need per simulation
reins_need = (
    federal_df.groupby('simulation')['reins_need']
    .sum()
    .reindex(sim_index, fill_value=0)
)

# Apply logic at the state level
def state_drawdown(row):
    contrib, bal = row['contribution'], row['nfip_balance']
    if contrib >= 0:
        return 0
    elif bal >= 0:
        return abs(min(0, bal + contrib))
    else:
        return abs(contrib)

state_balance_df['neg_draw'] = state_balance_df.apply(state_drawdown, axis=1)

# Then sum by simulation and state, then across states
total_neg = (
    state_balance_df
    .groupby(['simulation', 'STATEFP'])['neg_draw']
    .sum()
    .groupby('simulation')
    .sum()
    .reindex(sim_index, fill_value=0)
)

summary = pd.DataFrame({
    'simulation': sim_index,
    'Total_Reinsurance_Used': reins_need.values/steps,  # Federal pooled
    'Total_Negative_Balance': total_neg.values/steps    # State unpooled
})

# Stack simulation outcomes for each metric
bar_df = summary[['Total_Reinsurance_Used', 'Total_Negative_Balance']].copy()
bar_df = bar_df.rename(columns={
    'Total_Reinsurance_Used': 'Federal Pool',
    'Total_Negative_Balance': 'State Pools'
})

# Melt into long format
bar_long = bar_df.melt(var_name="Pool", value_name="Amount")
bar_long['Amount'] = bar_long['Amount']/1000000000

In [ ]:
# Normalize STATEFP to zero-padded 2-char strings
state_balance_df["STATEFP"] = state_balance_df["STATEFP"].astype(str).str.extract(r"(\d+)", expand=False).str.zfill(2)

df = state_balance_df.copy()
df["State"] = df["STATEFP"].map({
    '22': 'LA', '28': 'MS', '01': 'AL', '48': 'TX', '36': 'NY', '34': 'NJ',
    '29': 'MO', '40': 'OK', '54': 'WV', '38': 'ND', '19': 'IA', '27': 'MN',
    '42': 'PA', '12': 'FL', '06': 'CA'
})

# Filter to selected states
selected_states = ['LA', 'MS', 'AL', 'TX', 'NY', 'NJ', 'MO', 'OK', 'WV', 'ND', 'IA', 'MN', 'PA', 'FL', 'CA']
df = df[df["State"].isin(selected_states)]

# Add cumulative and yearly balance tracking
df["nfip_balance"] = df.groupby(["simulation", "State"])["contribution"].cumsum()/1000
df["yearly_debt"] = df["contribution"]  # Negative contributions = debt

# Filter to negative balance years only
neg_df = df[df["nfip_balance"] < 0]

# Pivot for plotting
pivot_df = (
    neg_df.groupby(["year", "State"])["nfip_balance"]
    .mean()
    .unstack(fill_value=0)
)

# Normalize by premium
premium_df = (
    df.groupby("State")["premium"]
    .mean()
    .replace(0, np.nan)
)

# Compute year-over-year added deficit
delta_neg_billion = pivot_df.diff().clip(upper=0).abs() / 1e6
pivot_df_billion = pivot_df / 1e6

# Clean normalization: replace inf/NaN from zero/NaN premiums
normalized_pivot_df = (
    pivot_df
    .divide(premium_df, axis=1)
    .replace([np.inf, -np.inf], np.nan)
    .fillna(0.0)
)

normalized_pivot_df = normalized_pivot_df * 100

# Normalize YOY added deficit (after cleaning)
delta_norm = (
    normalized_pivot_df
    .diff()
    .clip(upper=0)
    .abs()
)

# Ensure year index is numeric and sorted
for _df in [pivot_df, normalized_pivot_df, delta_neg_billion, pivot_df_billion, delta_norm]:
    _df.index = pd.to_numeric(_df.index, errors='coerce')
    _df.sort_index(inplace=True)

# Build colors dict from the union of all columns you might plot
all_cols = sorted(set(pivot_df.columns) |
                  set(normalized_pivot_df.columns) |
                  set(delta_neg_billion.columns) |
                  set(delta_norm.columns))
cmap = cm.get_cmap("Spectral", max(len(all_cols), 1))
colors_dict = {s: cmap(i) for i, s in enumerate(all_cols)}
colors_dict['ND'] = (1.0, 0.973, 0.676, 1.0)  # optional override

In [ ]:
sns.set_theme(style="ticks", font_scale=0.6)
plt.rcParams.update({
    "xtick.major.size": 2.5,
    "ytick.major.size": 2.5,
    "xtick.minor.size": 1.5,
    "ytick.minor.size": 1.5,
    "xtick.major.pad": 2.0,
    "ytick.major.pad": 2.0,
    "axes.linewidth": 0.5,
    "xtick.major.width": 0.5,  # add this
    "ytick.major.width": 0.5,  # add this
})

from matplotlib.gridspec import GridSpec, GridSpecFromSubplotSpec

def custom_stackplot(ax, df, title, ylabel, xlabel, label_text, label_pos, colors_dict):
    if df is None or df.empty or df.shape[1] == 0:
        ax.set_title(title)
        ax.set_xlabel(xlabel)
        ax.set_ylabel(ylabel)
        ax.text(label_pos[0], label_pos[1], label_text, transform=ax.transAxes,
                ha='left', va='top', fontsize=8, fontweight='bold')
        ax.text(0.5, 0.5, "No data", transform=ax.transAxes, ha='center', va='center', fontsize=11)
        return

    # Use the columns of THIS df, not a global list
    cols = list(df.columns)

    # Build a color list that matches the number of series exactly
    cmap_local = cm.get_cmap("Spectral", len(cols))
    if colors_dict is None:
        color_list = [cmap_local(i) for i in range(len(cols))]
    else:
        color_list = [colors_dict.get(c, cmap_local(i)) for i, c in enumerate(cols)]

    x = df.index
    y = [df[c].values for c in cols]

    # Unpack series with *y
    ax.stackplot(x, *y, labels=cols, colors=color_list, linewidth=0)
    ax.set_title(title)
    ax.set_xlabel(xlabel)
    ax.set_ylabel(ylabel)
    ax.text(label_pos[0], label_pos[1], label_text, transform=ax.transAxes,
            ha='left', va='top', fontsize=8, fontweight='bold')


# Shared figure
fig = plt.figure(figsize=(7, 5.25))

gs_root = GridSpec(nrows=1, ncols=2, figure=fig, wspace=0.15)

# LEFT: State line plots
gs_left = GridSpecFromSubplotSpec(n_rows, n_cols, subplot_spec=gs_root[0], hspace=0.4, wspace=0.4)

for i, state in enumerate(selected_states):
    r, c = divmod(i, n_cols)
    ax = fig.add_subplot(gs_left[r, c])

    df_sub = state_balance_stats[state_balance_stats["State"] == state]
    ax.fill_between(df_sub["year"], df_sub["min"]/1e3, df_sub["max"]/1e3, color="gray", alpha=0.2, label="Range")
    ax.fill_between(df_sub["year"], df_sub["q25"]/1e3, df_sub["q75"]/1e3, color="gray", alpha=0.4, label="IQR")
    ax.plot(df_sub["year"], df_sub["mean"]/1e3, color="black", linewidth=2, label="Mean")
    ax.axhline(0, color='black', linestyle='--')

    df_sub_hist = state_balance_by_year_hist[state_balance_by_year_hist["State"] == state]
    ax.plot(df_sub_hist["year"], df_sub_hist["nfip_balance"]/1e3, color='#B04743', linewidth=2, label='Historic')

    ax.set_title(f"{state}")
    ax.text(-0.05, 1.15, f"{chr(97 + i)})", transform=ax.transAxes,
            fontsize=8, fontweight='bold', va='top', ha='right')

    if i // n_cols == n_rows - 1:
        ax.set_xlabel("Year")

    if i // n_cols < n_rows - 1:
        ax.tick_params(labelbottom=False)

# Left-side shared legend (anchor to bottom of left half)
left_axes = [fig.axes[i] for i in range(len(selected_states))]
handles_l, labels_l = left_axes[0].get_legend_handles_labels()
fig.legend(handles_l, labels_l, loc="lower center",
           bbox_to_anchor=(0.29, -0.01), ncol=4, frameon=False)

# After gs_left is defined and all left axes are added
gs_left_bbox = gs_root[0].get_position(fig)
left_mid_x = (gs_left_bbox.x0 + gs_left_bbox.x1) / 2
left_top_y = gs_left_bbox.y1

fig.text(left_mid_x, left_top_y + 0.04, "State Pool Balance ($B)",
         ha='center', va='bottom', fontsize=8)

# RIGHT: 2x2 summary panels
panel_labels = ['p)', 'q)', 'r)', 's)']
label_offset = (-0.1, 1.10)

gs_right = GridSpecFromSubplotSpec(2, 2, subplot_spec=gs_root[1], hspace=0.4, wspace=0.3)

# (0,0) National time series
ax00 = fig.add_subplot(gs_right[0, 0])
ax00.fill_between(stats_by_year['year'], stats_by_year['min'], stats_by_year['max'],
                  alpha=0.2, color='gray', label='Range')
ax00.fill_between(stats_by_year['year'], stats_by_year['q25'], stats_by_year['q75'],
                  alpha=0.4, color='gray', label='IQR')
sns.lineplot(data=time_series_df_hist, x="year", y="total_balance",
             ax=ax00, color='#B04743', lw=2, label='Historic', legend=False)
sns.lineplot(data=stats_by_year, x='year', y='mean',
             ax=ax00, color='black', lw=2, label='Average')
ax00.set_title("Federal Pool ($B)")
ax00.text(label_offset[0], label_offset[1], panel_labels[0],
          transform=ax00.transAxes, fontsize=8, fontweight='bold')
ax00.set_ylabel("")
ax00.set_xlabel("Year")
ax00.legend(frameon=False)

# (1,0) Boxplots
ax10 = fig.add_subplot(gs_right[1, 0])
sns.boxplot(data=bar_long, x="Pool", y="Amount", ax=ax10,
            palette={"Federal Pool": "tab:blue", "State Pools": "tab:red"})
ax10.set_title("Reinsurance\nEst. Need ($B)")
ax10.set_ylabel("")
ax10.set_xlabel("")
ax10.text(label_offset[0], label_offset[1], panel_labels[1],
          transform=ax10.transAxes, fontsize=8, fontweight='bold')

label_offset = (-0.1, 1.15)

# (0,1) Cumulative gross deficit stackplot
ax01 = fig.add_subplot(gs_right[0, 1])
custom_stackplot(ax01, pivot_df_billion,
                 "Gross Debt ($M)",
                 "", "Year",
                 panel_labels[2], label_offset, colors_dict)

# (1,1) Premium-normalized deficit stackplot
ax11 = fig.add_subplot(gs_right[1, 1])
custom_stackplot(ax11, normalized_pivot_df,
                 "Premium-Relative\nDebt (%)",
                 "", "Year",
                 panel_labels[3], label_offset, colors_dict)

gs_right_bbox = gs_root[1].get_position(fig)
right_mid_x = (gs_right_bbox.x0 + gs_right_bbox.x1) / 2

legend_states = list(pivot_df_billion.columns)
handles_r = [plt.Line2D([0], [0], color=colors_dict[s], lw=4) for s in legend_states]
fig.legend(handles_r, legend_states,
           loc="lower center",
           bbox_to_anchor=(right_mid_x, -0.04),
           ncol=len(legend_states)//3,   # or a fixed int like ncol=5
           title="", fontsize=6, title_fontsize=6,
           frameon=False)

min_font = min(
    item.get_fontsize()
    for ax in fig.axes
    for item in (
        ax.get_xticklabels() + ax.get_yticklabels() +
        [ax.title, ax.xaxis.label, ax.yaxis.label]
    )
    if hasattr(item, 'get_fontsize')
)
print(f"Minimum font size in figure: {min_font:.1f}pt")

if save:
    plt.savefig(f"Plots/Fig1_{test_case}.pdf", dpi=500, bbox_inches='tight')
plt.show()